## 0. 제목 / 연구 질문 / 금지 해석
# NBJ-006 · Physics proxy를 더한 최종 비지도 모델 후보 비교

**실행 전 골격 — 저장 출력 없음.** NBJ-005 Raw4를 고정 기준으로 두고 3개 파생변수의 추가가 K-Means 공정 상태와 IF 이상도 구조·seed 안정성에 어떤 차이를 만드는지 탐색한다. H1: 순위·후보 구조가 달라질 수 있다. H2: 유용한 후보라면 seed가 바뀌어도 안정적이어야 한다. 어느 변화도 불량 예측 성능 향상을 뜻하지 않는다. 군집·후보는 정상/불량 판정이 아니다. 날짜별 기록과의 관계는 탐색적이다.

## 1. Imports / environment
실행 전 환경·패키지 버전을 표시한다. NBJ-005 저장 환경과 다르면 regression gate에서 중단한다. CPU thread 설정도 NBJ-005와 맞춘다.

In [ ]:
import os
for _name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[_name] = "4"
import hashlib, io, json, sys, zipfile
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import adjusted_rand_score, silhouette_score
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "experiments" / "NBJ").is_dir():
    raise RuntimeError("저장소 루트에서 Notebook을 실행하세요.")
HERE = ROOT / "experiments" / "NBJ" / "nbj-006"
BASE = ROOT / "experiments" / "NBJ" / "nbj-005"
META_PATH = BASE / "outputs" / "models" / "metadata.json"
MODEL_PATH = BASE / "outputs" / "models" / "pipeline.joblib"
BASE_ROWS = BASE / "outputs" / "inference_rows.csv"
QUALITY_PATH = ROOT / "experiments" / "NBJ" / "nbj-004" / "outputs" / "quality_daily.csv"
SEEDS = [0, 1, 2, 3, 42]
RTOL, ATOL = 1e-10, 1e-12
print({"python": sys.version, "executable": sys.executable, "pythonpath": os.environ.get("PYTHONPATH"),
       "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__})

## 2. NBJ-005 metadata 읽기
모든 baseline 숫자는 실제 metadata에서 확인한다. sys.executable과 PYTHONPATH를 NBJ-005 기록과 문자열로 정확 비교한다. 다르면 모델 fit 전에 중단한다.

In [ ]:
if not all(p.is_file() for p in (META_PATH, MODEL_PATH, BASE_ROWS)):
    raise FileNotFoundError("NBJ-005 metadata, bundle 또는 inference_rows.csv가 없습니다.")
meta = json.loads(META_PATH.read_text(encoding="utf-8"))
if sys.executable != meta["python_executable"]:
    raise RuntimeError(f'Python executable mismatch: {sys.executable!r} != {meta["python_executable"]!r}')
if os.environ.get("PYTHONPATH") != meta["pythonpath"]:
    raise RuntimeError(f'PYTHONPATH mismatch: {os.environ.get("PYTHONPATH")!r} != {meta["pythonpath"]!r}')
sha256 = lambda b: hashlib.sha256(b).hexdigest()
assert meta["verification_passed"] is True
assert sha256(MODEL_PATH.read_bytes()) == meta["bundle_sha256"]
assert sha256((BASE / "pipeline.py").read_bytes()) == meta["code_sha256"]
assert (np.__version__, pd.__version__, sklearn.__version__) == (
    meta["numpy"], meta["pandas"], meta["sklearn"])
assert sys.version == meta["python"]
assert int(meta["threads"]) == 4
assert meta["feature_order"] == ["weld force(bar)", "weld current(kA)", "weld Voltage(v)", "weld time(ms)"]
assert meta["train_cutoff"] == "2020-03-31" and (meta["train_rows"], meta["followup_rows"]) == (8470, 3469)
assert meta["kmeans"]["n_clusters"] == 2 and meta["seed"] == 42
assert meta["threshold_definition"].startswith("train score 95% linear quantile")
assert meta["score_definition"] == "-score_samples(IF scaler transform(X))"
RAW4 = list(meta["feature_order"])
ZIP_PATH = Path(meta["source"]["zip_path"])
print({"source": meta["source"], "kmeans": meta["kmeans"],
       "isolation_forest": meta["isolation_forest"], "threshold": meta["threshold"]})

## 3. 원본 hash 및 schema 검증
ZIP와 내부 Excel의 바이트 해시를 확인한다. 원본은 읽기만 한다. 품질 기록은 같은 Excel의 result 시트에만 의존한다.

In [ ]:
if not ZIP_PATH.is_file():
    raise FileNotFoundError(ZIP_PATH)
zip_bytes = ZIP_PATH.read_bytes()
assert sha256(zip_bytes) == meta["source"]["zip_sha256"]
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
    excel_bytes = archive.read(meta["source"]["zip_member"])
assert sha256(excel_bytes) == meta["source"]["excel_sha256"]
raw = pd.read_excel(io.BytesIO(excel_bytes), sheet_name=meta["source"]["sheet"])
quality_raw = pd.read_excel(io.BytesIO(excel_bytes), sheet_name="result")
assert raw.shape == (11939, 10) and list(raw.columns[6:]) == RAW4
assert len(quality_raw) == 23
assert raw[RAW4].notna().all().all()
assert np.isfinite(raw[RAW4].to_numpy(float)).all()
raw["excel_row"] = np.arange(2, len(raw) + 2)
raw["date"] = pd.to_datetime(raw["working time"], errors="raise").dt.strftime("%Y-%m-%d")
raw["input_id"] = [f'{meta["source"]["excel_sha256"]}:Raw data:{i}' for i in raw["excel_row"]]
assert raw["date"].nunique() == 9 and raw["excel_row"].is_unique
print({"raw_rows": len(raw), "result_rows": len(quality_raw), "dates": raw["date"].nunique()})

## 4. train/follow-up split 재현
마스크는 NBJ-005와 같은 날짜 기준이며 모든 feature set과 seed에서 공유한다.

In [ ]:
train_mask = raw["date"].le(meta["train_cutoff"]).to_numpy(bool)
split = np.where(train_mask, "train", "follow-up")
assert (int(train_mask.sum()), int((~train_mask).sum())) == (8470, 3469)
assert np.all(np.diff(raw["excel_row"].to_numpy()) == 1)
display(pd.DataFrame({"split": split}).value_counts().rename("rows"))

## 5. Raw 4 feature 확인
원본 이름·순서·단위를 그대로 사용한다. 반복 값과 특이 값은 삭제하지 않는다.

In [ ]:
display(raw[RAW4].describe().T)
display(raw.groupby("date", sort=True).size().rename("raw_rows"))

## 6. physics proxy 생성
전류 0 또는 비유한 결과가 있으면 멈춘다. R은 접촉저항 측정치가 아니고 VIt/I²t는 실제 열량이 아니다.

In [ ]:
force, current, voltage, weld_time = RAW4
if raw[current].eq(0).any():
    raise ValueError("Current 0 발견: R_proxy 분모 불가")
features = raw[RAW4].copy()
features["R_proxy"] = raw[voltage] / raw[current]
features["VI_t_proxy"] = raw[voltage] * raw[current] * raw[weld_time]
features["I2_t_proxy"] = raw[current] ** 2 * raw[weld_time]
DERIVED = ["R_proxy", "VI_t_proxy", "I2_t_proxy"]
if not np.isfinite(features.to_numpy(float)).all():
    raise ValueError("비유한 파생값 발견")
assert features.index.equals(raw.index)

## 7. derived feature 기술통계
진단은 train만 사용하며, Weld Time의 고유값과 빈도를 별도로 본다.

In [ ]:
train_features = features.loc[train_mask]
display(train_features.describe().T)
display(raw.loc[train_mask, weld_time].value_counts(dropna=False).sort_index().rename("train_rows"))

## 8. Pearson/Spearman redundancy 분석
상관이 높아도 feature를 이번 실험 중 임의 삭제하지 않는다. 파생변수끼리의 상관도 확인한다.

In [ ]:
pearson = train_features.corr(method="pearson")
spearman = train_features.corr(method="spearman")
display(pearson)
display(spearman)
display(spearman.loc[DERIVED, DERIVED])
fig, axes = plt.subplots(1, 2, figsize=(15, 6), layout="constrained")
for ax, matrix, title in zip(axes, (pearson, spearman), ("Train Pearson", "Train Spearman")):
    im = ax.imshow(matrix, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set(xticks=range(len(matrix)), yticks=range(len(matrix)),
           xticklabels=matrix.columns, yticklabels=matrix.index, title=title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    fig.colorbar(im, ax=ax, label="Correlation")
plt.show()

## 9. feature set 정의
명시적 순서를 고정한다. 새로운 sensor 정보라는 표현을 쓰지 않는다.

In [ ]:
FEATURE_SETS = {
    "FS0_RAW4": RAW4,
    "FS1_R": RAW4 + ["R_proxy"],
    "FS2_ENERGY": RAW4 + ["VI_t_proxy"],
    "FS3_I2T": RAW4 + ["I2_t_proxy"],
    "FS4_ALL": RAW4 + DERIVED,
}
assert [len(v) for v in FEATURE_SETS.values()] == [4, 5, 5, 5, 7]
assert all(v[:4] == RAW4 for v in FEATURE_SETS.values())
display(pd.DataFrame({"set": list(FEATURE_SETS), "features": list(FEATURE_SETS.values())}))

## 10. FS0 seed=42 NBJ-005 regression test
**Gate:** 이 셀이 실패하면 뒤 셀을 실행하지 않는다. 전체 행 ID/분할/군집/후보는 exact, 거리·점수·문턱은 NBJ-005 허용오차로 비교한다. 저장된 NBJ-005 bundle·결과는 읽기만 한다.

In [ ]:
def fit_k(set_name, seed):
    x = features[FEATURE_SETS[set_name]].to_numpy(float)
    scaler = StandardScaler().fit(x[train_mask])
    model = KMeans(**{**meta["kmeans"], "random_state": seed})
    model.fit(scaler.transform(x[train_mask]))
    z = scaler.transform(x)
    labels = model.predict(z)
    distance = np.linalg.norm(z - model.cluster_centers_[labels], axis=1)
    return {"scaler": scaler, "model": model, "labels": labels,
            "distance": distance, "train_z": z[train_mask]}

def fit_if(set_name, seed):
    x = features[FEATURE_SETS[set_name]].to_numpy(float)
    scaler = StandardScaler().fit(x[train_mask])
    model = IsolationForest(**{**meta["isolation_forest"], "random_state": seed})
    model.fit(scaler.transform(x[train_mask]))
    score = -model.score_samples(scaler.transform(x))
    calculated = float(np.quantile(score[train_mask], .95, method="linear"))
    threshold = float(meta["threshold"]) if (set_name, seed) == ("FS0_RAW4", 42) else calculated
    flag = score >= threshold
    return {"scaler": scaler, "model": model, "score": score,
            "threshold_calculated": calculated, "threshold": threshold, "flag": flag}

REGRESSION_GATE_PASSED = False
k_runs = {("FS0_RAW4", 42): fit_k("FS0_RAW4", 42)}
if_runs = {("FS0_RAW4", 42): fit_if("FS0_RAW4", 42)}
k0, i0 = k_runs[("FS0_RAW4", 42)], if_runs[("FS0_RAW4", 42)]
saved = pd.read_csv(BASE_ROWS)
assert len(saved) == len(raw)
assert np.array_equal(raw["input_id"].to_numpy(), saved["input_id"].to_numpy())
assert np.array_equal(raw["excel_row"].to_numpy(), saved["excel_row"].to_numpy())
assert np.array_equal(raw["date"].to_numpy(), saved["date"].to_numpy())
assert np.array_equal(split, saved["split"].to_numpy())
assert np.array_equal(k0["labels"], saved["cluster_id"].to_numpy())
assert np.allclose(k0["distance"], saved["centroid_distance_scaled"], rtol=RTOL, atol=ATOL)
assert np.allclose(i0["score"], saved["if_anomaly_score"], rtol=RTOL, atol=ATOL)
assert np.array_equal(i0["flag"], saved["candidate_flag"].to_numpy(bool))
assert np.isclose(i0["threshold_calculated"], meta["threshold"], rtol=RTOL, atol=ATOL)
assert np.array_equal(np.bincount(k0["labels"], minlength=2), 
                      np.bincount(saved["cluster_id"], minlength=2))
for which in ("train", "follow-up"):
    mask = split == which
    assert int(mask.sum()) == int((saved["split"] == which).sum())
    assert np.array_equal(np.bincount(k0["labels"][mask], minlength=2),
                          np.bincount(saved.loc[mask, "cluster_id"], minlength=2))
    assert int(i0["flag"][mask].sum()) == int(saved.loc[mask, "candidate_flag"].sum())
check = pd.DataFrame({"date": raw["date"], "cluster_id": k0["labels"], "flag": i0["flag"]})
new_group_check = check[["date", "cluster_id"]].copy()
old_group_check = saved[["date", "cluster_id"]].copy()
new_group_check["cluster_id"] = new_group_check["cluster_id"].astype("int64")
old_group_check["cluster_id"] = old_group_check["cluster_id"].astype("int64")
new_cluster = new_group_check.groupby(["date", "cluster_id"]).size()
old_cluster = old_group_check.groupby(["date", "cluster_id"]).size()
pd.testing.assert_series_equal(new_cluster, old_cluster)
new_if = check.groupby("date")["flag"].sum()
old_if = saved.groupby("date")["candidate_flag"].sum()
pd.testing.assert_series_equal(new_if, old_if, check_names=False)
REGRESSION_GATE_PASSED = True
print("NBJ-005 FS0 seed=42 regression gate: PASS")

## 11. K-Means multi-seed fitting
Gate 이후 같은 5개 set×5개 seed의 나머지 K-Means를 학습한다. K를 재탐색하지 않는다.

In [ ]:
assert REGRESSION_GATE_PASSED is True, "NBJ-005 regression gate failed or was not run"
assert ("FS0_RAW4", 42) in k_runs and ("FS0_RAW4", 42) in if_runs
for set_name in FEATURE_SETS:
    for seed in SEEDS:
        if (set_name, seed) not in k_runs:
            k_runs[(set_name, seed)] = fit_k(set_name, seed)
assert len(k_runs) == 25

## 12. K-Means stability 분석
Train 전체 silhouette, 분할·날짜별 크기/비율, seed 쌍 ARI와 NBJ-005 대비 ARI를 만든다. 새 set끼리 cluster ID 직접 일치율로 판단하지 않는다.

In [ ]:
k_rows, k_pairs, k_profiles = [], [], []
for set_name, cols in FEATURE_SETS.items():
    for seed in SEEDS:
        run = k_runs[(set_name, seed)]
        labels = run["labels"]
        sil = float(silhouette_score(run["train_z"], labels[train_mask]))
        for part in ("train", "follow-up"):
            mask = split == part
            for cluster in (0, 1):
                k_rows.append({"set": set_name, "seed": seed, "split": part,
                               "cluster_id": cluster, "rows": int(np.sum(labels[mask] == cluster)),
                               "train_silhouette": sil})
    for a, b in combinations(SEEDS, 2):
        k_pairs.append({"set": set_name, "seed_a": a, "seed_b": b,
                        "ari": adjusted_rand_score(k_runs[(set_name, a)]["labels"],
                                                   k_runs[(set_name, b)]["labels"])})
    labels42 = k_runs[(set_name, 42)]["labels"]
    for cluster in (0, 1):
        frame = features.loc[labels42 == cluster, cols]
        for col in cols:
            k_profiles.append({"set": set_name, "cluster_id": cluster, "feature": col,
                               "n": len(frame), "median": frame[col].median(),
                               "iqr": frame[col].quantile(.75) - frame[col].quantile(.25)})
k_summary = pd.DataFrame(k_rows)
k_seed_pairs = pd.DataFrame(k_pairs)
k_profile = pd.DataFrame(k_profiles)
daily_cluster_rows = []
for set_name in FEATURE_SETS:
    for seed in SEEDS:
        frame = pd.DataFrame({"date": raw["date"], "cluster": k_runs[(set_name, seed)]["labels"]})
        counts = frame.groupby(["date", "cluster"]).size()
        for (date, cluster), n in counts.items():
            daily_cluster_rows.append({"set": set_name, "seed": seed, "date": date,
                                       "cluster_id": cluster, "rows": int(n),
                                       "share_pct": 100 * n / int((raw["date"] == date).sum())})
daily_cluster = pd.DataFrame(daily_cluster_rows)
display(k_summary)
display(k_seed_pairs)
display(k_profile)
display(daily_cluster)

## 13. IF multi-seed fitting
FS0 anchor는 재사용하고 나머지 set×seed만 fit한다. 각각 자체 train 점수의 95% linear quantile을 쓴다.

In [ ]:
for set_name in FEATURE_SETS:
    for seed in SEEDS:
        if (set_name, seed) not in if_runs:
            if_runs[(set_name, seed)] = fit_if(set_name, seed)
assert len(if_runs) == 25
if_rows, if_daily_rows = [], []
for (set_name, seed), run in if_runs.items():
    for part in ("train", "follow-up"):
        mask = split == part
        if_rows.append({"set": set_name, "seed": seed, "split": part,
                        "threshold": run["threshold"], "candidate_n": int(run["flag"][mask].sum()),
                        "candidate_pct": float(run["flag"][mask].mean() * 100),
                        "score_median": float(np.median(run["score"][mask]))})
    daily = pd.DataFrame({"date": raw["date"], "score": run["score"], "flag": run["flag"]})
    for date, group in daily.groupby("date"):
        if_daily_rows.append({"set": set_name, "seed": seed, "date": date,
                              "candidate_n": int(group["flag"].sum()),
                              "candidate_pct": float(group["flag"].mean() * 100),
                              "score_median": float(group["score"].median())})
if_summary = pd.DataFrame(if_rows)
if_daily = pd.DataFrame(if_daily_rows)
display(if_summary)
display(if_daily)

## 14. IF score/candidate stability 분석
같은 set의 10개 seed 쌍에 대해 전체 행 score 순위와 후보 집합 겹침을 계산한다.

In [ ]:
def jaccard(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    union = int(np.logical_or(a, b).sum())
    return 1.0 if union == 0 else float(np.logical_and(a, b).sum() / union)

if_pairs = []
for set_name in FEATURE_SETS:
    for a, b in combinations(SEEDS, 2):
        left, right = if_runs[(set_name, a)], if_runs[(set_name, b)]
        if_pairs.append({"set": set_name, "seed_a": a, "seed_b": b,
                         "score_spearman": pd.Series(left["score"]).corr(pd.Series(right["score"]), method="spearman"),
                         "candidate_jaccard": jaccard(left["flag"], right["flag"])})
if_seed_pairs = pd.DataFrame(if_pairs)
display(if_seed_pairs)

## 15. FS0 vs physics feature set 비교
모두 seed 42의 **같은 행**을 비교한다. 유지·탈락·신규 후보는 실제 불량 변화가 아니다.

In [ ]:
anchor_k = k_runs[("FS0_RAW4", 42)]
anchor_if = if_runs[("FS0_RAW4", 42)]
fs0_compare = []
changed_masks = {}
for set_name in FEATURE_SETS:
    kr, ir = k_runs[(set_name, 42)], if_runs[(set_name, 42)]
    kept = anchor_if["flag"] & ir["flag"]
    lost = anchor_if["flag"] & ~ir["flag"]
    gained = ~anchor_if["flag"] & ir["flag"]
    changed_masks[set_name] = {"kept": kept, "lost": lost, "gained": gained}
    fs0_compare.append({"set": set_name,
                        "cluster_ari_vs_fs0": adjusted_rand_score(anchor_k["labels"], kr["labels"]),
                        "score_spearman_vs_fs0": pd.Series(anchor_if["score"]).corr(pd.Series(ir["score"]), method="spearman"),
                        "candidate_overlap_n": int(kept.sum()),
                        "candidate_jaccard_vs_fs0": jaccard(anchor_if["flag"], ir["flag"]),
                        "kept_n": int(kept.sum()), "lost_n": int(lost.sum()), "gained_n": int(gained.sum())})
fs0_comparison = pd.DataFrame(fs0_compare)
display(fs0_comparison)

## 16. 날짜별 exploratory quality relation
원본 result에는 생산·검사 수량 분모가 없다. 1,000 Raw 기록당 기재 불량 건수는 실제 불량률이 아니다. 3/27 미기록과 3/31 불완전 유형을 제외한다. 각 set의 seed 42 군집을 FS0 seed 42와 최대 행 overlap으로 정렬하고 mapping을 기록한다. ARI는 변경하지 않는다.

In [ ]:
EXPECTED_RESULT_COLUMNS = ["idx", "Machine_Name", "Item No", "working time", "defect", "defect type", "Unnamed: 6"]
if list(quality_raw.columns) != EXPECTED_RESULT_COLUMNS:
    raise ValueError("result schema changed; denominator meaning must be rechecked before quality analysis")
if not QUALITY_PATH.is_file():
    raise FileNotFoundError(QUALITY_PATH)
prior_quality = pd.read_csv(QUALITY_PATH).set_index("date")
raw_daily_rows = raw.groupby("date").size()
assert set(prior_quality.index) == set(raw_daily_rows.index)
assert np.array_equal(prior_quality.loc[raw_daily_rows.index, "raw_records"].to_numpy(), raw_daily_rows.to_numpy())
quality_raw["date"] = pd.to_datetime(quality_raw["working time"], errors="raise").dt.strftime("%Y-%m-%d")
quality = quality_raw.groupby("date").agg(recorded_defects=("defect", "sum"),
                                          recorded_types=("defect type", "nunique"))
quality["raw_rows"] = raw_daily_rows.reindex(quality.index)
assert "2020-03-27" not in quality.index
assert int(quality.loc["2020-03-31", "recorded_types"]) == 2
assert np.array_equal(quality["recorded_types"].to_numpy(), prior_quality.loc[quality.index, "recorded_types"].to_numpy())
assert np.array_equal(quality["recorded_defects"].to_numpy(), prior_quality.loc[quality.index, "recorded_sum_partial"].to_numpy())
quality = quality.loc[quality["recorded_types"].eq(3)].copy()
assert len(quality) == 7
quality["defects_per_1000_raw_records"] = 1000 * quality["recorded_defects"] / quality["raw_rows"]
assert np.allclose(quality["defects_per_1000_raw_records"], prior_quality.loc[quality.index, "complete_sum_per1000records"], rtol=RTOL, atol=ATOL)
print("Quality denominator: Raw daily row count, not confirmed inspected/product count; seven complete recorded dates")
alignment_rows, aligned_cluster_by_set = [], {}
reference = anchor_k["labels"]
for set_name in FEATURE_SETS:
    labels = k_runs[(set_name, 42)]["labels"]
    identity_n = int(np.sum(labels == reference))
    swap_n = int(np.sum((1 - labels) == reference))
    mapping = {0: 0, 1: 1} if identity_n >= swap_n else {0: 1, 1: 0}
    aligned = np.array([mapping[int(label)] for label in labels], dtype=int)
    aligned_cluster_by_set[set_name] = aligned
    alignment_rows.append({"set": set_name, "seed": 42, "mapping": mapping,
                           "identity_overlap_n": identity_n, "swap_overlap_n": swap_n,
                           "selected_overlap_n": int(np.sum(aligned == reference)),
                           "tie_rule": "identity"})
cluster_alignment = pd.DataFrame(alignment_rows)
display(cluster_alignment)
quality_rows = []
for set_name in FEATURE_SETS:
    ir = if_runs[(set_name, 42)]
    frame = pd.DataFrame({"date": raw["date"], "score": ir["score"],
                          "flag": ir["flag"], "aligned_cluster": aligned_cluster_by_set[set_name]})
    daily = frame.groupby("date").agg(score_median=("score", "median"),
                                      candidate_share=("flag", "mean"),
                                      aligned_cluster1_share=("aligned_cluster", "mean"))
    daily[["candidate_share", "aligned_cluster1_share"]] *= 100
    joined = daily.join(quality[["recorded_defects", "defects_per_1000_raw_records"]], how="inner")
    for indicator in ("score_median", "candidate_share", "aligned_cluster1_share"):
        for target in ("recorded_defects", "defects_per_1000_raw_records"):
            quality_rows.append({"set": set_name, "indicator": indicator, "target": target,
                                 "n_dates": len(joined),
                                 "spearman": joined[indicator].corr(joined[target], method="spearman")})
quality_exploratory = pd.DataFrame(quality_rows)
display(quality_exploratory)

## 17. candidate changed-row 분석
FS0 대비 탈락·신규 후보를 같은 Raw 센서·proxy의 중앙값/IQR로 설명한다. 행별 정답은 없다.

In [ ]:
changed_rows = []
changed_profiles = []
for set_name in FEATURE_SETS:
    for status, mask in changed_masks[set_name].items():
        selected = raw.loc[mask, ["excel_row", "date"]].copy()
        selected["set"] = set_name
        selected["status"] = status
        changed_rows.append(selected)
        for col in features.columns:
            values = features.loc[mask, col]
            changed_profiles.append({"set": set_name, "status": status, "feature": col,
                                     "n": len(values), "median": values.median(),
                                     "iqr": values.quantile(.75) - values.quantile(.25)})
changed_row_table = pd.concat(changed_rows, ignore_index=True)
changed_profile_table = pd.DataFrame(changed_profiles)
display(changed_profile_table)

## 18. 최종 비교표
자동 우승자를 정하지 않는다. train_candidate_pct는 train 95% 분위수 문턱 적용의 sanity check라 대략 5%가 되며, 모델 우열 판단에 사용하지 않는다. 해석 장점·한계 및 중복성 진단은 출력 뒤 사람이 근거를 적는다.

In [ ]:
def pair_stat(table, col):
    return table.groupby("set")[col].agg(["median", "min"]).rename(
        columns={"median": col + "_median", "min": col + "_min"})

comparison = pd.DataFrame(index=FEATURE_SETS)
comparison.index.name = "set"
comparison["feature_n"] = [len(v) for v in FEATURE_SETS.values()]
comparison["train_silhouette"] = (k_summary.query("seed == 42 and split == 'train'")
                                   .drop_duplicates("set").set_index("set")["train_silhouette"])
comparison = comparison.join(pair_stat(k_seed_pairs, "ari"))
comparison = comparison.join(pair_stat(if_seed_pairs, "score_spearman"))
comparison = comparison.join(pair_stat(if_seed_pairs, "candidate_jaccard"))
comparison = comparison.join(fs0_comparison.set_index("set")[["cluster_ari_vs_fs0",
    "score_spearman_vs_fs0", "candidate_overlap_n", "candidate_jaccard_vs_fs0",
    "kept_n", "lost_n", "gained_n"]])
for part, label in (("train", "train_candidate_pct"), ("follow-up", "followup_candidate_pct")):
    comparison[label] = if_summary.query("seed == 42 and split == @part").set_index("set")["candidate_pct"]
comparison["train_candidate_pct_role"] = "sanity check only; not for model ranking"
comparison["redundancy_max_abs_spearman"] = [
    0.0 if len(cols) == 4 else spearman.loc[[c for c in cols if c in DERIVED], [x for x in cols if x not in DERIVED]].abs().max().max()
    for cols in FEATURE_SETS.values()
]
comparison["changed_candidate_conditions"] = "검토 후 changed_profile_table을 근거로 기술"
comparison["interpretation_advantage"] = "검토 전"
comparison["interpretation_limit"] = "검토 전"
display(comparison)

## 19. 결론 작성용 빈 Markdown cell

**실행 후 사람이 작성:**
- FS0 regression gate:
- H1 관찰:
- H2 관찰:
- 최종 후보 및 근거:
- FS0 유지 여부:
- 날짜별 기록과의 탐색적 관계:
- 예상 밖 결과·실패:

## 20. Limitations
제품별 정상/불량 label과 검사시점이 없고 날짜별 기록은 일부 누락된다. 반복 센서 조합과 같은 데이터에서의 feature 비교·선정으로 일반화 및 예측 성능을 입증할 수 없다. 파생 proxy는 새 독립 센서가 아니며 접촉저항·실제 열량 측정치도 아니다. K=2는 NBJ-005에서 잠정 선택된 공정 구분이므로 물리적 운전모드 확정이나 품질 등급이 아니다.

## 21. Next action
이 실행 전 계획과 코드를 검토한 후에만 별도 실행 단계에서 regression gate부터 진행한다. Gate 실패 시 설정을 바꾸지 않고 원인을 기록한다. 모든 결과가 나온 뒤 비교표를 사람이 검토해 최종 비지도 모델 후보를 결정한다.